# 1. Import Data

## 1.1. Import Libraries

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from scipy.stats.stats import kurtosis
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

## 1.2 Import Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Change pwd
os.chdir("/content/drive/MyDrive/Colab Notebooks/DSML/Project")

In [ ]:
#Import CSV
df = pd.read_csv('donors_descriptive.csv')
df.head()

<a name='index'></a>
## 1.3. Set Index

In [ ]:
# Set CONTROL_NUMBER as the index — it is a unique identifier with no analytical value
df = df.set_index('CONTROL_NUMBER')
df.head()

## 1.4 Categorize Data
Here we'll categorize our data into smaller sets in order to better organize our Data Pre-Processing

### Sociodemographics

In [ ]:
#Lists of columns grouped by the following categories defined
#for the data provided

###Sociodemographics##
NON_CATEGORICAL_DATA = ["DONOR_AGE", "CHILDREN"]
CATEGORICAL_DATA = ["DONOR_GENDER", "HOME_OWNER", "SES", "URBANICITY", "INCOME_GROUP", "WEALTH_RATING",]
ALL_COLS_SOCIO_DEMOGRAPH = ["DONOR_AGE", "DONOR_GENDER", "HOME_OWNER", "INCOME_GROUP", "SES", "URBANICITY", "WEALTH_RATING", "CHILDREN"]

#Final Dataframe for Sociodemographics data.
df_socioDemograph = df[ALL_COLS_SOCIO_DEMOGRAPH]

### Neighborhood-level Indicators

In [ ]:
# Define the two feature groups
FINANCIAL_COLS = [
    'MEDIAN_HOME_VALUE',
    'MEDIAN_HOUSEHOLD_INCOME',
    'PCT_OWNER_OCCUPIED',
    'PER_CAPITA_INCOME'
]

MILITARY_COLS = [
    'PCT_ATTRIBUTE1',  # % active military (male)
    'PCT_ATTRIBUTE2',  # % veterans (male)
    'PCT_ATTRIBUTE3',  # % Vietnam veterans
    'PCT_ATTRIBUTE4'   # % WW2 veterans
]

ALL_COLS = FINANCIAL_COLS + MILITARY_COLS

#Final Dataframe for Neighboorhood-level indicators data.
df_neighborhood_levels = df[ALL_COLS]

### Campaign Interaction & Donation Behaviour

In [ ]:
NUMERIC_CAMP_BEHAV_COLS = [
    "LIFETIME_PROM",
    "LIFETIME_CARD_PROM",
    "RECENT_RESPONSE_COUNT",
    "RECENT_CARD_RESPONSE_COUNT",
    "NUMBER_PROM_12",
    "CARD_PROM_12",
    "RECENT_RESPONSE_PROP",
    "RECENT_CARD_RESPONSE_PROP",
    "MONTHS_SINCE_FIRST_GIFT",
    "MONTHS_SINCE_LAST_GIFT",
    "MONTHS_SINCE_LAST_PROM_RESP",
    "LIFETIME_GIFT_COUNT",
]

ALL_COLS_CAMP_BEHAV = [
    "LIFETIME_PROM",
    "LIFETIME_CARD_PROM",
    "RECENT_RESPONSE_COUNT",
    "RECENT_CARD_RESPONSE_COUNT",
    "NUMBER_PROM_12",
    "CARD_PROM_12",
    "RECENT_RESPONSE_PROP",
    "RECENT_CARD_RESPONSE_PROP",
    "PEP_STAR",
    "RECENT_STAR_STATUS",
    "MONTHS_SINCE_FIRST_GIFT",
    "MONTHS_SINCE_LAST_GIFT",
    "MONTHS_SINCE_LAST_PROM_RESP",
    "LIFETIME_GIFT_COUNT",
    "FREQUENCY_STATUS_97NK",
    "RECENCY_STATUS_96NK",
]

#Final Dataframe for Campaign Interaction & Donation Behaviour Data
df_campaigns = df[ALL_COLS_CAMP_BEHAV]
df_campaigns

### Donation Amounts

In [ ]:
# filter for donation variables

DONATION_AMTS_COLS = [
    'RECENT_AVG_GIFT_AMT',
    'LAST_GIFT_AMT',
    'LIFETIME_GIFT_AMOUNT',
    'LIFETIME_MAX_GIFT_AMT',
    'LIFETIME_MIN_GIFT_AMT',
    'FILE_CARD_GIFT',
    'RECENT_AVG_CARD_GIFT_AMT'
]

df_donations = df[DONATION_AMTS_COLS]
df_donations.head()

<a name='duplicates'></a>
## 1.5. Check for Duplicates

In [ ]:
# Check for duplicated rows in the entire dataset
n_duplicates = df.duplicated().sum()
print(f'Number of duplicated rows: {n_duplicates}')

# 2. Explore Data

## 2.1. Basic Exploration

### SocioDemographics

In [ ]:
#Check data types
df_socioDemograph.info()

In [ ]:
#Describe non categorical columns

#Show medians
print(f"Median values:\n {df_socioDemograph[NON_CATEGORICAL_DATA].median()}")
print(f"\nMode values:\n {df_socioDemograph[CATEGORICAL_DATA].mode().T}")
df_socioDemograph[CATEGORICAL_DATA + NON_CATEGORICAL_DATA].describe(include='all').T

In [ ]:
#Percentage of missing values
df_socioDemograph.isnull().sum() / len(df_neighborhood_levels) * 100

### Neighborhood-level Indicators

In [ ]:
# Shape, data types, and a preview of the selected features
print('Shape:', df_neighborhood_levels[ALL_COLS].shape)
print('\nData types:')
print(df_neighborhood_levels[ALL_COLS].dtypes)
print('\nFirst rows:')
df_neighborhood_levels[ALL_COLS].head()

In [ ]:
# Information of the featured dataframe
df_neighborhood_levels[ALL_COLS].info()

In [ ]:
# Missing value count and percentage per feature
missing = df_neighborhood_levels[ALL_COLS].isnull().sum()
missing_pct = (missing / len(df_neighborhood_levels) * 100).round(2)

missing_df_neighborhood_levels = pd.DataFrame({'Missing Count': missing, 'Missing (%)': missing_pct})
print(missing_df_neighborhood_levels)

In [ ]:
# Descriptive statistics
df_neighborhood_levels[ALL_COLS].describe().T

In [ ]:
## DELETE THE CELL BEFORE (DUPLICATE)
# Descriptive statistics
df_neighborhood_levels[ALL_COLS].describe().T.round(2)

**Observations**

*   Negative Values where there shouldn't be any, investigate further


In [ ]:
# Check for negative values — these are physically implausible for percentages and monetary fields
for col in ALL_COLS:
    neg_count = (df_neighborhood_levels[col] < 0).sum()
    if neg_count > 0:
        print(f'{col}: {neg_count} negative values  |  min = {df_neighborhood_levels[col].min():.2f}')

**Observation:** Several features contain negative values that are implausible — e.g., negative `MEDIAN_HOME_VALUE` or negative percentages. Could be data entry errors and will be treated as invalid (set to NaN) during cleaning. (confirm action)

In [ ]:
# Skewness per feature
print('Skewness:')
print(df_neighborhood_levels[ALL_COLS].skew().round(3))

**Observation:** `MEDIAN_HOME_VALUE`, `PER_CAPITA_INCOME`, and especially `PCT_ATTRIBUTE1` are strongly right-skewed. A power transform will be applied to reduce skewness before clustering.

**Justification**: K-Means relies on Euclidean distance to measure similarity between donors. When a feature is heavily skewed, a small number of very high-value donors will dominate the distance calculations, pulling clusters toward them. The result is that clusters end up shaped by the outliers rather than by the actual structure of the data.

In [ ]:
# Skewness and kurtosis per feature
shape_stats = df_neighborhood_levels[ALL_COLS].agg(['skew', 'kurt']).T.round(3)
shape_stats.columns = ['Skewness', 'Kurtosis']
print(shape_stats)

**Observation:** Several features show strong right skewness (|skew| > 1), particularly `MEDIAN_HOME_VALUE`, `PER_CAPITA_INCOME`, and `PCT_ATTRIBUTE1` — these asymmetric distributions would distort Euclidean distances in clustering and justify applying a Yeo-Johnson power transform. High kurtosis values on the same features confirm heavy tails with extreme values, reinforcing the decision to cap outliers at the 99th percentile before transforming.

### Campaing and Donation Behaviour

In [ ]:
# Check overall info, including data types (16 variables)
df_campaigns.info()

In [ ]:
# Check missing values
cpgn_missing = df_campaigns.isna().sum()
cpgn_missing_pct = (cpgn_missing/len(df_campaigns)*100).round(1)

print(pd.DataFrame({'missing Count': cpgn_missing, 'Missing %': cpgn_missing_pct}))


In [ ]:
# Check records with at least a missing value (total of 3830)
cpgn_missing_rows = df_campaigns[df_campaigns.isna().any(axis=1)]
cpgn_missing_rows

In [ ]:
# Check for basic statistics
df_campaigns.describe().T # df_campaigns.describe(include =['O']) for categorical variables only (O=Object) which there are none in df_campaigns

### Donation Amounts

In [ ]:
# check for missing values

df_donations.isna().sum()

In [ ]:
# check for negative values

# create a boolean dataframe where true indicates a negative value
negative_values_df = df_donations.select_dtypes(include=['number']) < 0

# sum the true values (which are treated as 1) for each column
negative_counts = negative_values_df.sum()

# display the counts of negative values
display(negative_counts[negative_counts > 0])

In [ ]:
# identify the data type of each column

df_donations.info()
df_donations.shape

In [ ]:
# describe numerical variables for key stats
df_donations.describe().T

In [ ]:
# check for kurtosis

kurtosis = df_donations.kurtosis()
display(kurtosis)

In [ ]:
# check for skewness

skewness = df_donations.skew()
display(skewness)

## 2.2 Visual Exploration

### SocioDemographics

Since Donor age has a very high standard deviation, it's best to create a distribution plot to better visualize outliers, skewness and assymetry of this attribute

In [ ]:
#Distribution plot for Donor Age
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distributions', fontsize=14, fontweight='bold')

#Define different bins for each attribute
bins_dict = {
    "DONOR_AGE": 15,
    "CHILDREN": range(0, 10),
}

for ax, col in zip(axes, NON_CATEGORICAL_DATA):
    sns.histplot(df_socioDemograph[col].dropna(), kde=True, ax=ax, color='steelblue', bins=bins_dict[col])
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('bins')

plt.tight_layout()
plt.show()

**Observation:** We can check that for every attribute there are outliers that should be taken care of.
For **DONOR_AGE** although the mean value is around 59 Years old, we can verify that there are 2 main groups of ages that form around that mean:
*   ~ 40 and 50 years old
*   ~ 65 and 75 years old

For **CHILDREN** the distribution is near uniform which is not normal in this context since the tendency is for people to have none or around 1-2 children. This very even distribution might tell us that the data on this column has been sampled or balanced.

Plots for Categorical Data

In [ ]:
#Absolute Values
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

for ax, col in zip(axes.flatten(), CATEGORICAL_DATA):
    sns.countplot(df_socioDemograph, x=col, ax=ax)
    ax.set_title(str(col), fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

### Neighborhood-level Indicators

In [ ]:
# Distribution plots — Group A: Neighborhood Financial Indicators
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Group A — Neighborhood Financial Indicators: Distributions', fontsize=14, fontweight='bold')

for ax, col in zip(axes, FINANCIAL_COLS):
    sns.histplot(df_neighborhood_levels[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots — Group B: Military Affiliation
mil_labels = ['PCT_ATTR1\n(Active Military)', 'PCT_ATTR2\n(Veterans)', 'PCT_ATTR3\n(Vietnam Vets)', 'PCT_ATTR4\n(WW2 Vets)']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Group B — Military Affiliation: Distributions', fontsize=14, fontweight='bold')

for ax, col, label in zip(axes, MILITARY_COLS, mil_labels):
    sns.histplot(df_neighborhood_levels[col].dropna(), kde=True, ax=ax, color='darkorange')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots to visualise spread and outliers — both groups
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_neighborhood_levels[FINANCIAL_COLS].boxplot(ax=axes[0])
axes[0].set_title('Group A — Financial Indicators', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

df_neighborhood_levels[MILITARY_COLS].boxplot(ax=axes[1])
axes[1].set_title('Group B — Military Affiliation', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Campaign & Donation Behaviour

In [ ]:
# Check for Skewness and Kurtosis
num_df = df_campaigns.select_dtypes(include=['number'])
summary_stats_cpgns = num_df.agg(['skew', 'kurtosis']).T.round(2)
print(summary_stats_cpgns)

In [ ]:
# Build box plots to understand distributions and itentify outliers
num_cols_cpgns = df_campaigns.select_dtypes(include=['number']).columns

for col in num_cols_cpgns:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df_campaigns[col])
    plt.title(col)
    plt.show()

In [ ]:
# Visualize relationship between PEP_STAR e RECENT_STAR_STATUS
sns.scatterplot(data = df_campaigns, x = 'PEP_STAR', y= 'RECENT_STAR_STATUS')

### Donation Amounts

In [ ]:
# check for correlation

# select only numeric columns for correlation calculation
numeric_df_donations = df_donations.select_dtypes(include=['number'])

# calculate the Pearson correlation matrix
correlation_matrix = numeric_df_donations.corr()

# display the correlation matrix as a heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Pearson Correlation Matrix of Donation Variables')
plt.show()

In [ ]:
# select only numerical columns for plotting histograms
numerical_cols = df_donations.select_dtypes(include=['number']).columns

# determine the number of rows and columns for the subplots (a good heuristic is to use sqrt(num_cols) for both rows and columns)
num_numerical_cols = len(numerical_cols)
num_rows = int(num_numerical_cols / 3) + (num_numerical_cols % 3 > 0)
num_cols = 3

plt.figure(figsize=(num_cols * 5, num_rows * 4))

for i, col in enumerate(numerical_cols):
    plt.subplot(num_rows, num_cols, i + 1)
    sns.histplot(df_donations[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8)) = plt.subplots(4, 2, figsize=(18, 10))

sns.boxplot(ax=ax1, data=df_donations, x='RECENT_AVG_GIFT_AMT')
sns.boxplot(ax=ax2, data=df_donations, x='LAST_GIFT_AMT')
sns.boxplot(ax=ax3, data=df_donations, x='LIFETIME_GIFT_AMOUNT')
sns.boxplot(ax=ax4, data=df_donations, x='LIFETIME_MAX_GIFT_AMT')
sns.boxplot(ax=ax5, data=df_donations, x='LIFETIME_MIN_GIFT_AMT')
sns.boxplot(ax=ax6, data=df_donations, x='FILE_CARD_GIFT')
sns.boxplot(ax=ax7, data=df_donations, x='RECENT_AVG_CARD_GIFT_AMT')

ax8.set_visible(False)

plt.suptitle('Boxplots', fontsize=16)
plt.tight_layout()
plt.show()

## 2.3 In-depth Exploration

### SocioDemographics

In [ ]:
#Correlation Categorical Data
df_analysis = df.copy()

#Filter for columns
df_analysis = df_analysis[CATEGORICAL_DATA]

#Mapping of categorical values into numeric value
map_gender = {'U': 0, 'F': 1, 'M': 2}
map_homeowner = {'U': 0, 'H': 1}
map_urbanicity = {'?': 0, 'C': 1, 'R': 2, 'S':3, 'T': 4, 'U': 5}
map_ses = {'?': 0}

df_analysis['DONOR_GENDER'] = df_analysis['DONOR_GENDER'].map(map_gender)
df_analysis['HOME_OWNER'] = df_analysis['HOME_OWNER'].map(map_homeowner)
df_analysis['URBANICITY'] = df_analysis['URBANICITY'].map(map_urbanicity)
df_analysis['SES'] = df_analysis['SES'].replace('?', 0)

print(df_analysis.info())

#Normalize all columns to int
df_analysis['DONOR_GENDER'] = df_analysis['DONOR_GENDER'].astype('Int64')
df_analysis['HOME_OWNER'] = df_analysis['HOME_OWNER'].astype('Int64')
df_analysis['SES'] = pd.to_numeric(df_analysis['SES'], errors='coerce').astype('Int64')
df_analysis['URBANICITY'] = df_analysis['URBANICITY'].astype('Int64')
df_analysis['INCOME_GROUP'] = df_analysis['INCOME_GROUP'].round().astype('Int64')
df_analysis['WEALTH_RATING'] = df_analysis['WEALTH_RATING'].round().astype('Int64')

df_analysis

In [ ]:
#Plot Heatmap to check for correlation
plt.figure(figsize=(6,5))

df_analysis = df_analysis.dropna()

#First, scale values using MinMaxScaler
scaler = MinMaxScaler()
df_analysis = pd.DataFrame(
    scaler.fit_transform(df_analysis),
    columns=df_analysis.columns,
    index=df_analysis.index
)

#Plot heatmap with the spearman method. The spearman method
#is more indicated to use when checking for the correlation
#of categorigal ordinal variables

sns.heatmap(
    df_analysis.corr(method='spearman'),
    annot=True, fmt='.2f', cmap='coolwarm',
    square=True, linewidths=0.5,
    vmin=-1, vmax=1
)

plt.title('Categorical Columns Correlation')
plt.tight_layout()
plt.show()

**Observations:** From the correlation heatmap, one can conclude the following:

*   There is a moderate-to-high correlation between Wealth Rating and SES, and a moderate-to-low correlation between Income Group and SES. Both correlations are negative which at least tells us the following information: The higher the Wealth Rating/Income Group, the lower the SES attribute is.
*   There is some similaraty between Income Group and Wealth rating, but not enough to consider them as equal, or "merge" them into one.
*   There is a low correlation between the Home Owner and Income Group attributes, telling us that people who are home owners can belong to higher income groups but not necessarily.



### Neighborhood-level Indicators

In [ ]:
# Correlation heatmap — Group A
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    df_neighborhood_levels[FINANCIAL_COLS].corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    ax=axes[0], square=True, linewidths=0.5
)
axes[0].set_title('Group A — Financial Indicators\nCorrelation Matrix', fontweight='bold')

# Correlation heatmap — Group B
sns.heatmap(
    df_neighborhood_levels[MILITARY_COLS].corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    ax=axes[1], square=True, linewidths=0.5
)
axes[1].set_title('Group B — Military Affiliation\nCorrelation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

### Campaign & Donation Behaviour

In [ ]:
# Check for correlations
def correlation_heatmap():
  #Check correrations
  num_df = df_campaigns.select_dtypes(include=['number'])
  corr = num_df.corr(method='spearman')

  mask = np.triu(np.ones_like(corr, dtype=bool))

  plt.figure(figsize=(16, 10))
  sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm')
  plt.title('Spearman Correlation - Numeric Variables')
  plt.show()

correlation_heatmap()

**Key observations:**
1) We see some strong correlationsa above 7.0 between between the overall promotion variables and the card promotions since card is a subset of total promotion.

2) It's normal that the number of months since the first gift is highly correlation with the number of lifetime promotions (0.85) and the number of lifetime gifts (0.81), showing that donors that have been donating for longer also have been impacted by more promotions and also have donated more times.

3) Response proportions, for both overall promotions and card only, have also high correlations above 0.75 with the number of responses since it reflects the ration between responses and promotions (the higher the number of responses, the higher that propotion)

# 3. Preprocess Data

## 3.1. Data Cleaning

### 3.1.1. Outliers

#### SocioDemographics

**DONOR AGE**

In [ ]:
### DONOR AGE ###
df_decimal_ages = df_socioDemograph.copy()

#Split column into two
df_decimal_ages[["DONOR_AGE", "DECIMAL_AGE"]] = df_decimal_ages["DONOR_AGE"].astype(str).str.split('.', expand=True)

#Get entries where age has decimal values
df_decimal_ages = df_decimal_ages[(df_decimal_ages["DECIMAL_AGE"] != '0') & (df_decimal_ages["DONOR_AGE"] != 'nan')]
df_decimal_ages.info()
df_decimal_ages.head(5)

There are 138 entries that are outliers. Since this is a very low percentage of records it is safe to assign them with the mean value of age

In [ ]:
#Convert real age to int
df_decimal_ages["DONOR_AGE"] = df_decimal_ages["DONOR_AGE"].astype(int)

#Get mean age from main dataframe
mean_age = df_socioDemograph['DONOR_AGE'].dropna().astype(int).mean()

#Replace in column
df_decimal_ages["DONOR_AGE"] = mean_age.round().astype(int)

print(df_decimal_ages["DONOR_AGE"].dtype)
df_decimal_ages.head(5)

In [ ]:
#Join new ages to main dataframe 'df_socioDemograph'
df_socioDemograph.update(df_decimal_ages["DONOR_AGE"])

**CHILDREN**

There are entries with negative numbers and floats. Let's apply abs() and np.ceil() to round them to nearest integer

In [ ]:
df_children = df_socioDemograph.copy()

#Fill missing rows with rounded down mean
children_median = df_children["CHILDREN"].median()
df_children["CHILDREN"].fillna(children_median, inplace=True)

#Apply abs() and round()
df_children["CHILDREN"] = df_children["CHILDREN"].abs().apply(np.ceil)

df_children['CHILDREN'].value_counts()

An amount of 7 children can be classified as an outlier as an outlier in this case since it doesn't respect the distribution at all. There are no entries with 5 or 6 children so this is a highly unlikely scenario

In [ ]:
df_children['CHILDREN'].clip(lower=0, upper=4, axis=0, inplace=True)
df_children['CHILDREN'].value_counts()

In [ ]:
#Join with main dataframe
df_socioDemograph['CHILDREN'].update(df_children['CHILDREN'])

#Convert to correct data type INT
df_socioDemograph['CHILDREN'] = df_socioDemograph['CHILDREN'].astype(int)

**INCOME GROUP**

In [ ]:
df_income = df_socioDemograph.copy()

#Check Excess Kurtosis of column Income Group
print(f"Kurtosis: {df_income['INCOME_GROUP'].kurt()}")

#Outliers
sns.boxplot(df_income, x='INCOME_GROUP')

Excess Kurtosis level is inferior to zero. This tells us that there aren't many outliers in this data. Having checked the previous boxplot and knowing that the range of income group must be between 1 and 7, we can safely assume the outliers are values out of that range. In this ccase, checking the maximum and minimum values of Income group we can see that these are values that are negative and of type float. Let's fix them

In [ ]:
#Check for values out of range. In this case it was very easy to spot them
#with a simple value_counts()
df_income['INCOME_GROUP'].value_counts(dropna=False)

In [ ]:
# Clip values to a range of 1 and 7
df_income['INCOME_GROUP'].clip(lower=1, upper=7, axis=0, inplace=True)
df_income['INCOME_GROUP'].value_counts(dropna=False)

#### Neighborhood-level Indicators

Negative values were identified in monetary and percentage features. These will be replaced with NaN so they can be handled alongside other missing values.

Two separate outlier treatments are applied:

1.   Negatives → NaN: Justified by domain impossibility. Setting them to NaN (rather than zero or some floor) is the better approach because zero would be a fabricated meaningful value, whereas NaN correctly signals "we don't know."

2.   99th percentile cap: Rather than removing extreme high-end values (which would discard rows), the values are clipped. This is a conservative approach — it limits the distortion of long tails on features like `MEDIAN_HOME_VALUE` without data loss.


In [ ]:
# Select the 8 relevant features
df_clean = df_neighborhood_levels[ALL_COLS].copy()

# Replace negative values with NaN — negatives are nonsensical for all 8 features
for col in ALL_COLS:
    n_neg = (df_clean[col] < 0).sum()
    df_clean.loc[df_clean[col] < 0, col] = np.nan
    if n_neg > 0:
        print(f'{col}: {n_neg} negative values replaced with NaN')

In [ ]:
# Cap extreme outliers at the 99th percentile to reduce the effect of long tails
# Relevant for MEDIAN_HOME_VALUE and PER_CAPITA_INCOME
for col in ALL_COLS:
    cap = df_clean[col].quantile(0.99)
    n_capped = (df_clean[col] > cap).sum()
    df_clean[col] = df_clean[col].clip(upper=cap)
    if n_capped > 0:
        print(f'{col}: {n_capped} values capped at 99th pct ({cap:.1f})')

#### Campaign & Donation Behaviour

For numeric variables we identified outliers and used clipping to fix them.

In [ ]:
# Define outliers using the IQR method
cpgn_vars1 = ['LIFETIME_PROM','LIFETIME_CARD_PROM', 'RECENT_RESPONSE_COUNT', 'RECENT_CARD_RESPONSE_COUNT', 'NUMBER_PROM_12', 'CARD_PROM_12', 'RECENT_RESPONSE_PROP', 'RECENT_CARD_RESPONSE_PROP', 'MONTHS_SINCE_FIRST_GIFT', 'MONTHS_SINCE_LAST_GIFT', 'MONTHS_SINCE_LAST_PROM_RESP', 'LIFETIME_GIFT_COUNT','FREQUENCY_STATUS_97NK']


cpgns_outliers_results = []

for var in cpgn_vars1:
    Q1  = df_campaigns[var].quantile(0.25)
    Q3  = df_campaigns[var].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_campaigns.query(f"{var} < @lower_bound or {var} > @upper_bound")

    cpgns_outliers_results.append({
        "Variable": var,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outliers": len(outliers)
    })

cpgns_outliers_summary_table = pd.DataFrame(cpgns_outliers_results)
print(cpgns_outliers_summary_table.to_string(index=False))


In [ ]:
# Fix outliers using clipping
bounds = {}

for var in cpgn_vars1:
    Q1  = df_campaigns[var].quantile(0.25)
    Q3  = df_campaigns[var].quantile(0.75)
    IQR = Q3 - Q1

    bounds[var] = {
        'lower': Q1 - 1.5 * IQR,
        'upper': Q3 + 1.5 * IQR
    }


for var in cpgn_vars1:
    lower = bounds[var]['lower']
    upper = bounds[var]['upper']

    df_campaigns[var] = df_campaigns[var].clip(lower, upper)

In [ ]:
# Verify remaining outliers after clipping
check_results = []

for _, row in cpgns_outliers_summary_table.iterrows():
    var = row["Variable"]
    lower = row["Lower Bound"]
    upper = row["Upper Bound"]

    outliers = df_campaigns.query(f"{var} < @lower or {var} > @upper")

    check_results.append({
        "Variable": var,
        "Remaining Outliers": len(outliers)
    })

pd.DataFrame(check_results)

In [ ]:
# Look at stats after removing outliers
df_cpgn_vars1 = df_campaigns[['LIFETIME_PROM','LIFETIME_CARD_PROM', 'RECENT_RESPONSE_COUNT', 'RECENT_CARD_RESPONSE_COUNT', 'NUMBER_PROM_12', 'CARD_PROM_12', 'RECENT_RESPONSE_PROP', 'RECENT_CARD_RESPONSE_PROP', 'MONTHS_SINCE_FIRST_GIFT', 'MONTHS_SINCE_LAST_GIFT', 'MONTHS_SINCE_LAST_PROM_RESP', 'LIFETIME_GIFT_COUNT','FREQUENCY_STATUS_97NK']]
df_cpgn_vars1.describe().T

In [ ]:
# Look at the visual impact using box plots
num_cols_cpgns = df_campaigns.select_dtypes(include=['number']).columns

for col in num_cols_cpgns:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df_campaigns[col])
    plt.title(col)
    plt.show()

After removing the outilers we don't see any data points beyond the box plots' whiskers.

#### Donation Amounts
As observed in 2.1. Basic Exploration, there are high levels of kurtosis, well above 100 in many cases, which indicates the presence of many outliers in the data. This is also confirmed by observing the BoxPlots in 2.2 Visual Exploration.
To fix this, we'll identify outliers using IQR Methodology and clip those values.



In [ ]:
# define outliers using the IQR methodology

donations_outliers_results = []

for var in DONATION_AMTS_COLS:
    Q1  = df_donations[var].quantile(0.25)
    Q3  = df_donations[var].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_donations.query(f"{var} < @lower_bound or {var} > @upper_bound")

    donations_outliers_results.append({
        "Variable": var,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outliers": len(outliers)
    })

donations_outliers_summary_table = pd.DataFrame(donations_outliers_results)
print(donations_outliers_summary_table.to_string(index=False))

In [ ]:
# fix outliers using clipping
bounds = {}

for var in DONATION_AMTS_COLS:
    Q1  = df_donations[var].quantile(0.25)
    Q3  = df_donations[var].quantile(0.75)
    IQR = Q3 - Q1

    bounds[var] = {
        'lower': Q1 - 1.5 * IQR,
        'upper': Q3 + 1.5 * IQR
    }


for var in DONATION_AMTS_COLS:
    lower = bounds[var]['lower']
    upper = bounds[var]['upper']

    df_donations[var] = df_donations[var].clip(lower, upper)

In [ ]:
# verify outliers after clipping
check_results = []

for _, row in donations_outliers_summary_table.iterrows():
    var = row["Variable"]
    lower = row["Lower Bound"]
    upper = row["Upper Bound"]

    outliers = df_donations.query(f"{var} < @lower or {var} > @upper")

    check_results.append({
        "Variable": var,
        "Remaining Outliers": len(outliers)
    })

pd.DataFrame(check_results)

### 3.1.2. Missing Values

#### SocioDemographics

**DONOR AGE**

**COURSE OF ACTION**: Recover with random values respecting the shape of the current distribution using resampling

In [ ]:
def random_impute_skewed(df, col):
    known = df[col].dropna()
    missing_mask = df[col].isna()
    n_missing = missing_mask.sum()

    # Sample randomly FROM the known values (respects actual shape)
    random_values = known.sample(n=n_missing, replace=True).values

    df.loc[missing_mask, col] = random_values
    df[col] = df[col].astype('Int64')
    return df

df_ages = df_socioDemograph.copy()

df_ages = random_impute_skewed(df_ages, 'DONOR_AGE')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df_socioDemograph["DONOR_AGE"], kde=True, ax=axes[0], color='steelblue', bins=10)
sns.histplot(df_ages["DONOR_AGE"], kde=True, ax=axes[1], color='orange', bins=10)
# axes[1].set_title("DONOR_AGE")
axes[0].legend()
axes[1].legend()

plt.suptitle('Distribution Before vs After Random Value Imputation')
plt.tight_layout()
plt.show()

**Conclusions**: As per the distribution plots, one can observe that the distribution is nearly identical after data imputation, maintaining the integrity of the data

In [ ]:
#Join cleaned data with main dataframe
df_socioDemograph['DONOR_AGE'].update(df_ages['DONOR_AGE'])

df_socioDemograph['DONOR_AGE'] = df_socioDemograph['DONOR_AGE'].astype(int)

DONOR GENDER

In [ ]:
#Create new dataframe to handle donor genders freely
df_donor_gender = df_socioDemograph.copy()

#Obtain mode of DONOR_GENDER
mode_gender = df_socioDemograph['DONOR_GENDER'].mode(0)[0]

#Fill missing values with the mode
df_donor_gender['DONOR_GENDER'].fillna(mode_gender, inplace=True)

#Check for missing values filling
df_donor_gender['DONOR_GENDER'].value_counts(dropna=False)

In [ ]:
#Join back to main 'df_socioDemograph' dataframe with cleaned data
df_socioDemograph['DONOR_GENDER'].update(df_donor_gender['DONOR_GENDER'])

INCOME GROUP

To recover missing values, we'll use he KNN algorithm utilizing the three columns with the most correlation: HomeOwner and SES Recover data using KNN Imputation

In [ ]:
def knn_elbow_method(x, y):

  # Test K values from 1 to 15
  k_range = range(1, 15)
  error_rates = []


  for k in k_range:
      knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='nan_euclidean')
      scores = cross_val_score(knn, x, y, cv=5, scoring='accuracy')
      error_rates.append(1 - scores.mean())

  # Plot elbow curve
  plt.figure(figsize=(10, 5))
  plt.plot(k_range, error_rates, marker='o', linestyle='--', color='steelblue')
  plt.xlabel('Number of Neighbors (K)')
  plt.ylabel('Error Rate')
  plt.title('Elbow Method for Optimal K')
  plt.xticks(k_range)
  plt.grid(True)
  plt.show()

#Mapping of categorical values into numeric value
map_homeowner = {'U': 0, 'H': 1}
map_ses = {'?': 0}

df_income['HOME_OWNER'] = df_income['HOME_OWNER'].map(map_homeowner)
df_income['SES'] = df_income['SES'].replace('?', 0)

print(df_income.info())

#Normalize all columns to int
df_income['HOME_OWNER'] = df_income['HOME_OWNER'].astype('Int64')
df_income['SES'] = pd.to_numeric(df_income['SES'], errors='coerce').astype('Int64')

error_rates = []

df_known = df_income.dropna(subset=['INCOME_GROUP'])

#Let's determine the optimal amount of k_neighbors using the elbow method
x = df_known[['SES', 'WEALTH_RATING', 'HOME_OWNER']]
y = df_known['INCOME_GROUP'].astype(int)

knn_elbow_method(x, y)

**Observation:** Observing the chart, we can conclude the optimal values are 9 or 10 n_neighbors to use for KNN. We'll go with n_neighbors=9 since

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('imputer', KNNImputer(n_neighbors=9, weights='distance', metric='nan_euclidean'))
])

imputed = pipeline.fit_transform(df_income[["INCOME_GROUP", "SES", "WEALTH_RATING", "HOME_OWNER"]])

scaler = pipeline.named_steps['scaler']
imputed_original = scaler.inverse_transform(imputed)

df_imputed = pd.DataFrame(imputed_original, columns=["INCOME_GROUP", "SES", "WEALTH_RATING", "HOME_OWNER"], index=df_socioDemograph.index)

#Apply rounding where if bigger or equal to .5 round up, else round down
df_imputed["INCOME_GROUP"] = df_imputed["INCOME_GROUP"].round()

In [ ]:
df_imputed.info()
df_imputed

In [ ]:
#Absolute Values
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(df_socioDemograph, x='INCOME_GROUP', ax=axes[0])
sns.countplot(df_imputed, x='INCOME_GROUP', ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
#Merge with main dataframe with cleaned data
df_socioDemograph.update(df_imputed['INCOME_GROUP'])

#update main dataframe
df_socioDemograph['INCOME_GROUP'] = df_socioDemograph['INCOME_GROUP'].astype(int)

SES

Since percent of missing values is quite low (~2%), we'll apply
the mode of the variable to fill them. Well'l apply this to
entries with '?' assigned

In [ ]:
df_ses = df_socioDemograph[["SES"]].copy()

print(df_ses.columns)

mode_ses = df_ses["SES"].mode(0)[0]
print(mode_ses)
#Fill NA
df_ses.fillna(mode_ses, inplace=True)

#Replace '?' with mode
df_ses.loc[df_ses["SES"] == '?'] = mode_ses

df_ses.value_counts(dropna=False)

In [ ]:
#Update main dataframe with cleaned values
df_socioDemograph['SES'].update(df_ses["SES"])

#Assign correct datatype
df_socioDemograph['SES'] = df_socioDemograph['SES'].astype(int)

URBANICITY

We'll simply assing the missing values to the category '?' which we can use to represent entries that we do not know the nature of the donor's location

In [ ]:
df_urbanicity = df_socioDemograph.copy()

df_urbanicity['URBANICITY'].value_counts(dropna=False)

In [ ]:
df_urbanicity['URBANICITY'] = df_urbanicity['URBANICITY'].fillna('?')

df_urbanicity['URBANICITY'].value_counts()

In [ ]:
#Assign correct data type and update main dataframe
df_urbanicity['URBANICITY'] = df_urbanicity['URBANICITY'].astype(str)
print(f"Data Type: {df_urbanicity['URBANICITY'].dtype}")

df_socioDemograph['URBANICITY'].update(df_urbanicity['URBANICITY'])

#### Neighbourhood-levels Indicator

In [ ]:
# Missing value counts after cleaning negatives
#(This was done in the Outliers cleanup)
print('Missing values after replacing negatives:')
print(df_clean.isnull().sum())

In [ ]:
# Impute with the median — robust to skewed distributions
for col in ALL_COLS:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

print('Missing values after median imputation:')
print(df_clean.isnull().sum())
print(f'\nFinal shape: {df_clean.shape}')

**Observation**: Median imputation is preferred because the distributions remain right-skewed even after outlier capping (as shown by the skewness values in Section 2.2). The median is a more robust central tendency measure for skewed data, as it is less sensitive to the residual asymmetry in the distribution compared to the mean.

#### Campaign & Donation Behaviour

In [ ]:
#Check for Missing Values
missing_campaigns = df_campaigns.isna().sum()
missing_campaigns_pct = (missing_campaigns/100).round(2)

print(pd.DataFrame({'Missing Count': missing_campaigns, 'Missing %': missing_campaigns_pct}))


In [ ]:
# Replace missing values for numeric variables using the median (excluding month variables)
df_campaigns.fillna({
    'LIFETIME_PROM': df_campaigns['LIFETIME_PROM'].median(),
    'LIFETIME_CARD_PROM': df_campaigns['LIFETIME_CARD_PROM'].median(),
    'RECENT_RESPONSE_COUNT': df_campaigns['RECENT_RESPONSE_COUNT'].median(),
    'RECENT_CARD_RESPONSE_COUNT': df_campaigns['RECENT_CARD_RESPONSE_COUNT'].median(),
    'NUMBER_PROM_12': df_campaigns['NUMBER_PROM_12'].median(),
    'CARD_PROM_12': df_campaigns['CARD_PROM_12'].median(),
    'RECENT_RESPONSE_PROP': df_campaigns['RECENT_RESPONSE_PROP'].median(),
    'RECENT_CARD_RESPONSE_PROP': df_campaigns['RECENT_CARD_RESPONSE_PROP'].median(),
    'LIFETIME_GIFT_COUNT': df_campaigns['LIFETIME_GIFT_COUNT'].median(),
    'FREQUENCY_STATUS_97NK': df_campaigns['FREQUENCY_STATUS_97NK'].median(),
    }, inplace = True)

Regarding the number of months since the last promotion response and number of months since the last gift we aligned these two variables that should be the same by filling out the nulls from one variable using the other one, or median of the second one if both columns are missing. Further, for the number of months since the first gift, used the last gift info to fill out missing values.

In [ ]:
# Fix missing values for month variables aligning all of them
df_campaigns['MONTHS_SINCE_LAST_GIFT'].fillna(df_campaigns['MONTHS_SINCE_LAST_PROM_RESP'], inplace=True)
df_campaigns['MONTHS_SINCE_LAST_GIFT'].fillna(df_campaigns['MONTHS_SINCE_LAST_GIFT'].median(), inplace=True)
df_campaigns['MONTHS_SINCE_LAST_PROM_RESP'].fillna(df_campaigns['MONTHS_SINCE_LAST_GIFT'], inplace=True)
df_campaigns['MONTHS_SINCE_FIRST_GIFT'].fillna(df_campaigns['MONTHS_SINCE_LAST_GIFT'], inplace=True)

RECENT_STAR_STATUS and PEP_STAR
If RECENT_STAR_STATUS, that represents whether a person donated in 3 consecutive promotions in the last 4 years, is 1 then PEP_STAR should also be 1 since it reflects that achievement of start status ever

In [ ]:
# Fix missing value for binary variables that should be aligned
df_campaigns['RECENT_STAR_STATUS'].fillna(0, inplace=True)
df_campaigns['PEP_STAR'].fillna(df_campaigns['RECENT_STAR_STATUS'], inplace=True)

In [ ]:
# Check missing values after cleanup
missing_campaigns = df_campaigns.isna().sum()
missing_campaigns_pct = (missing_campaigns/100).round(2)

print(pd.DataFrame({'Missing Count': missing_campaigns, 'Missing %': missing_campaigns_pct}))

**Observation:**
We left the variable RECENCY_STATUS_96NK's missing values for now since we will take care of it under the Data Transformation section.

#### Donation Amounts

In [ ]:
# substitute missing values for median (numerical variables)

for column in df_donations.select_dtypes(include=['number']).columns:
    median_value = df_donations[column].median()
    df_donations[column] = df_donations[column].fillna(median_value)

# verify that there are no more missing values in numerical columns
df_donations.isna().sum()

## 3.2. Data Transformation

### 3.2.1. Misclassifications & Incoherencies

#### SocioDemographics

**HOME OWNER**

In [ ]:
#Create copy of main dataframe
df_home_owners = df_socioDemograph.copy()

#Query for underage home owners (Age <=17 AND Home Owner == 'H') and set them as
#U (UNKNOWN)
df_home_owners["HOME_OWNER"].loc[(df_home_owners['DONOR_AGE'] <= 17) & (df_home_owners['HOME_OWNER'] == 'H')] = 'U'

In [ ]:
#Join cleaned data with main dataframe
df_socioDemograph['HOME_OWNER'].update(df_home_owners["HOME_OWNER"])

#Assign correct datatype
df_socioDemograph['HOME_OWNER'] = df_socioDemograph['HOME_OWNER'].astype(str)

#### Campaign & Donation Behaviour

Check for:
1) negatives or decimal in counts
2) card variables > overall variables AND lifetime > recent > 12 months
3) Proportions below 0 and above 1
4) Binary variables not 0 or 1 (rounding and clipping)
5) PEP_STAR = 0 AND RECENT_PEP_STAR = 1
6) Months since last month vs. last gift TBD vs. last gift
7) Recency_Status_96NK (will be taken care under the New Variables variables)


**STEP 1: COUNT VARIABLES**

In [ ]:
#Verify negative and decimal values for count variables that should countain only positive integers
cpgn_int = [
    'LIFETIME_PROM',
    'LIFETIME_CARD_PROM',
    'RECENT_RESPONSE_COUNT',
    'RECENT_CARD_RESPONSE_COUNT',
    'NUMBER_PROM_12',
    'CARD_PROM_12',
    'MONTHS_SINCE_FIRST_GIFT',
    'MONTHS_SINCE_LAST_GIFT',
    'MONTHS_SINCE_LAST_PROM_RESP',
    'LIFETIME_GIFT_COUNT',
    'FREQUENCY_STATUS_97NK'
]

((df_campaigns[cpgn_int] < 0) | ((df_campaigns[cpgn_int] > 0) & (df_campaigns[cpgn_int] < 1 ))).sum()

In [ ]:
# Correct for negative and decimal values
df_campaigns[cpgn_int] = (
    df_campaigns[cpgn_int]
    .clip(lower=0)   # replaces negatives with 0
    .round(0)        # rounds decimals
    .astype(int)     # converts to integer type
)

In [ ]:
# Verify negative and decimal values after fix
((df_campaigns[cpgn_int] < 0) | ((df_campaigns[cpgn_int] > 0) & (df_campaigns[cpgn_int] < 1 ))).sum()

In [ ]:
# Checking for inconsistencies in number of promotions
card12_over_promos12 = (df_campaigns['CARD_PROM_12'] > df_campaigns['NUMBER_PROM_12']).sum()
card12_over_lifetimecard = (df_campaigns['CARD_PROM_12'] > df_campaigns['LIFETIME_CARD_PROM']).sum()
lifetimecard_over_lifetime = (df_campaigns['LIFETIME_CARD_PROM'] > df_campaigns['LIFETIME_PROM']).sum()
promos12_over_lifetime = (df_campaigns['NUMBER_PROM_12'] > df_campaigns['LIFETIME_PROM']).sum()
lifetimegifts_over_lifetimepromos = (df_campaigns['LIFETIME_GIFT_COUNT'] > df_campaigns['LIFETIME_PROM']).sum()
lastgift_over_lastresp = (df_campaigns['MONTHS_SINCE_LAST_GIFT'] > df_campaigns['MONTHS_SINCE_LAST_PROM_RESP']).sum()
lastgift_over_firstgift = (df_campaigns['MONTHS_SINCE_LAST_GIFT'] > df_campaigns['MONTHS_SINCE_FIRST_GIFT']).sum()
frequency_over_lifetime = (df_campaigns['FREQUENCY_STATUS_97NK'] > df_campaigns['LIFETIME_PROM']).sum()

metrics = {
    "card12_over_promos12": card12_over_promos12,
    "card12_over_lifetimecard": card12_over_lifetimecard,
    "lifetimecard_over_lifetime": lifetimecard_over_lifetime,
    "promos12_over_lifetime": promos12_over_lifetime,
    "lifetimegifts_over_lifetimepromos": lifetimegifts_over_lifetimepromos,
    "lastgift_over_lastresp": lastgift_over_lastresp,
    "lastgift_over_firstgift": lastgift_over_firstgift,
    "frequency_over_lifetime": frequency_over_lifetime
}

for k, v in metrics.items():
    print(f"{k:35s}: {v}")

In [ ]:
## Fix inconsistencies

# There are no records where the number of card promotions is higher than the total number of promotions in the last 12 months, so no fix required

# Fixing lifetime card promotions when lower than card promotions in the last 12 months
cond1 = df_campaigns['CARD_PROM_12'] > df_campaigns['LIFETIME_CARD_PROM']
df_campaigns.loc[cond1, 'LIFETIME_CARD_PROM'] = df_campaigns.loc[cond1, 'CARD_PROM_12']

# Fixing lifetime promotions based on the max between (a) lifetime card promotions, (2) promotions over the last 12 months, and (3) lifetime gifts
cond2 = (df_campaigns['LIFETIME_CARD_PROM'] > df_campaigns['LIFETIME_PROM']) | \
          (df_campaigns['NUMBER_PROM_12'] > df_campaigns['LIFETIME_PROM']) | \
           (df_campaigns['LIFETIME_GIFT_COUNT'] > df_campaigns['LIFETIME_PROM'])

cols_cond2 = ['LIFETIME_CARD_PROM', 'NUMBER_PROM_12', 'LIFETIME_GIFT_COUNT']
df_campaigns.loc[cond2, 'LIFETIME_PROM'] = df_campaigns.loc[cond2,cols_cond2].max(axis=1)

# Fixing months since last gift when higher than months since last promo response
cond3 = df_campaigns['MONTHS_SINCE_LAST_GIFT'] > df_campaigns['MONTHS_SINCE_LAST_PROM_RESP']
df_campaigns.loc[cond3, 'MONTHS_SINCE_LAST_GIFT'] = df_campaigns.loc[cond3, 'MONTHS_SINCE_LAST_PROM_RESP']

# Fixing months since first gift when lower than last gift
cond4 = df_campaigns['MONTHS_SINCE_LAST_GIFT'] > df_campaigns['MONTHS_SINCE_FIRST_GIFT']
df_campaigns.loc[cond4, 'MONTHS_SINCE_FIRST_GIFT'] = df_campaigns.loc[cond4, 'MONTHS_SINCE_LAST_GIFT']



In [ ]:
# Verify results from fixes
card12_over_promos12 = (df_campaigns['CARD_PROM_12'] > df_campaigns['NUMBER_PROM_12']).sum()
card12_over_lifetimecard = (df_campaigns['CARD_PROM_12'] > df_campaigns['LIFETIME_CARD_PROM']).sum()
lifetimecard_over_lifetime = (df_campaigns['LIFETIME_CARD_PROM'] > df_campaigns['LIFETIME_PROM']).sum()
promos12_over_lifetime = (df_campaigns['NUMBER_PROM_12'] > df_campaigns['LIFETIME_PROM']).sum()
lifetimegifts_over_lifetimepromos = (df_campaigns['LIFETIME_GIFT_COUNT'] > df_campaigns['LIFETIME_PROM']).sum()
lastgift_over_lastresp = (df_campaigns['MONTHS_SINCE_LAST_GIFT'] > df_campaigns['MONTHS_SINCE_LAST_PROM_RESP']).sum()
lastgift_over_firstgift = (df_campaigns['MONTHS_SINCE_LAST_GIFT'] > df_campaigns['MONTHS_SINCE_FIRST_GIFT']).sum()
frequency_over_lifetime = (df_campaigns['FREQUENCY_STATUS_97NK'] > df_campaigns['LIFETIME_PROM']).sum()

metrics = {
    "card12_over_promos12": card12_over_promos12,
    "card12_over_lifetimecard": card12_over_lifetimecard,
    "lifetimecard_over_lifetime": lifetimecard_over_lifetime,
    "promos12_over_lifetime": promos12_over_lifetime,
    "lifetimegifts_over_lifetimepromos": lifetimegifts_over_lifetimepromos,
    "lastgift_over_lastresp": lastgift_over_lastresp,
    "lastgift_over_firstgift": lastgift_over_firstgift,
    "frequency_over_lifetime": frequency_over_lifetime
}

for k, v in metrics.items():
    print(f"{k:35s}: {v}")

**STEP 2: PROPORTION VARIABLES**

In [ ]:
# Check for proportion variables with values below 0 and above 1
outliers_resp_prop = ((df_campaigns['RECENT_RESPONSE_PROP'] < 0) | (df_campaigns['RECENT_RESPONSE_PROP'] > 1)).sum()
negative_resp_prop = (df_campaigns['RECENT_RESPONSE_PROP'] < 0).sum()
over1_resp_prop = (df_campaigns['RECENT_RESPONSE_PROP'] > 1).sum()

outliers_card_resp_prop = ((df_campaigns['RECENT_CARD_RESPONSE_PROP'] < 0) | (df_campaigns['RECENT_CARD_RESPONSE_PROP'] > 1)).sum()
negative_card_resp_prop = (df_campaigns['RECENT_CARD_RESPONSE_PROP'] < 0).sum()
over1_card_resp_prop = (df_campaigns['RECENT_CARD_RESPONSE_PROP'] > 1).sum()


print(f"'RECENT_RESPONSE_PROP': \n Total inconsistencies: {outliers_resp_prop} of which {negative_resp_prop} are negative and {over1_card_resp_prop} are over 1")
print("\n")
print(f"'RECENT_CARD_RESPONSE_PROP': \n Total inconsistencies: {outliers_card_resp_prop} of which {negative_card_resp_prop} are negative and {over1_card_resp_prop} are over 1")




In [ ]:
# Fix inconsistencies by clipping to 0 and 1
df_campaigns['RECENT_RESPONSE_PROP'] = df_campaigns['RECENT_RESPONSE_PROP'].clip(0, 1)
df_campaigns['RECENT_CARD_RESPONSE_PROP'] = df_campaigns['RECENT_CARD_RESPONSE_PROP'].clip(0, 1)

In [ ]:
# Verify results from fix
outliers_resp_prop = ((df_campaigns['RECENT_RESPONSE_PROP'] < 0) | (df_campaigns['RECENT_RESPONSE_PROP'] > 1)).sum()
outliers_card_resp_prop = ((df_campaigns['RECENT_CARD_RESPONSE_PROP'] < 0) | (df_campaigns['RECENT_CARD_RESPONSE_PROP'] > 1)).sum()

print(f"'RECENT_RESPONSE_PROP': \n Total inconsistencies: {outliers_resp_prop}")
print("\n")
print(f"'RECENT_CARD_RESPONSE_PROP': \n Total inconsistencies: {outliers_card_resp_prop}")
print("\n")

df_campaigns[['RECENT_RESPONSE_PROP','RECENT_CARD_RESPONSE_PROP']].describe()

**STEP 3: BINARY VARIABLES**

In [ ]:
# Possible Range is binary (0 or 1)
outliers_star_status = ((df_campaigns['PEP_STAR'] != 0) & (df_campaigns['PEP_STAR'] != 1)).sum()
outliers1_star_status = ((df_campaigns['PEP_STAR'] < 0) | (df_campaigns['PEP_STAR'] > 1)).sum()
outliers2_star_status = ((df_campaigns['PEP_STAR'] > 0) & (df_campaigns['PEP_STAR'] < 1)).sum()

outliers_recent_status = ((df_campaigns['RECENT_STAR_STATUS'] != 0) & (df_campaigns['RECENT_STAR_STATUS'] != 1)).sum()
outliers1_recent_status = ((df_campaigns['RECENT_STAR_STATUS'] < 0) | (df_campaigns['RECENT_STAR_STATUS'] > 1)).sum()
outliers2_recent_status = ((df_campaigns['RECENT_STAR_STATUS'] > 0) & (df_campaigns['RECENT_STAR_STATUS'] < 1)).sum()

print(f"'PEP_STAR': \n Total inconsistencies: {outliers_star_status} of which {outliers1_star_status} are out of the range [0,1], and {outliers2_star_status} are decimal between 0 and 1")
print("\n")
print(f"'RECENT_STAR_STATUS': \n Total inconsistencies: {outliers_recent_status} of which {outliers1_recent_status} are out of the range [0,1], and {outliers2_recent_status} are decimal between 0 and 1")
print("\n")

df_campaigns[['PEP_STAR','RECENT_STAR_STATUS']].describe()


In [ ]:
# Fix using clipping and results
df_campaigns['PEP_STAR'] = df_campaigns['PEP_STAR'].clip(0, 1)
df_campaigns['RECENT_STAR_STATUS'] = df_campaigns['RECENT_STAR_STATUS'].clip(0, 1)

outliers_star_status = ((df_campaigns['PEP_STAR'] != 0) & (df_campaigns['PEP_STAR'] != 1)).sum()
outliers_recent_status = ((df_campaigns['RECENT_STAR_STATUS'] != 0) & (df_campaigns['RECENT_STAR_STATUS'] != 1)).sum()

print(f"'PEP_STAR': \n Total inconsistencies: {outliers_star_status}")
print("\n")
print(f"'RECENT_STAR_STATUS': \n Total inconsistencies: {outliers_recent_status}")
print("\n")

df_campaigns[['PEP_STAR','RECENT_STAR_STATUS']].describe()

In [ ]:
# Flag inconsistencies where PEP_STAR is 1 and RECENT_STAR_STATUS is 0
PEPvsRECENT = (df_campaigns['RECENT_STAR_STATUS'] > df_campaigns['PEP_STAR']).sum()

print(f"Total inconsistencies: {PEPvsRECENT}")

In [ ]:
# Fix and verify results
cond3 = df_campaigns['RECENT_STAR_STATUS'] > df_campaigns['PEP_STAR']
df_campaigns.loc[cond3, 'PEP_STAR'] = df_campaigns.loc[cond3, 'RECENT_STAR_STATUS']

PEPvsRECENT = (df_campaigns['RECENT_STAR_STATUS'] > df_campaigns['PEP_STAR']).sum()

print(f"Total inconsistencies: {PEPvsRECENT}")

df_campaigns[['PEP_STAR','RECENT_STAR_STATUS']].value_counts()

In [ ]:
# Check data types after fixes
df_campaigns.info()

In [ ]:
# Fix data types and confirm results by setting binary variables as integeter
df_campaigns['PEP_STAR'] = df_campaigns['PEP_STAR'].astype(int)
df_campaigns['RECENT_STAR_STATUS'] = df_campaigns['RECENT_STAR_STATUS'].astype(int)
df_campaigns.info()

In [ ]:
# Statistics from Final Results
df_campaigns.describe().T

In [ ]:
df_campaigns.head(30)

#### Donation Amounts

In [ ]:
# substitute negative values for zero and round decimals

# columns_to_process = ['LIFETIME_GIFT_COUNT', 'RECENT_AVG_GIFT_AMT', 'LAST_GIFT_AMT', 'LIFETIME_GIFT_AMOUNT', 'LIFETIME_MAX_GIFT_AMT', 'LIFETIME_MIN_GIFT_AMT', 'FILE_CARD_GIFT', 'RECENT_AVG_CARD_GIFT_AMT']
df_donations.loc[:, DONATION_AMTS_COLS] = (
    df_donations[DONATION_AMTS_COLS]
    .clip(lower=0)   # replaces negatives with 0
    .round(0)        # rounds decimals
    .astype(int)     # converts to integer type
)

# verify negative and non-integer (decimal) values

print("Number of negative values per column:")
display((df_donations[DONATION_AMTS_COLS] < 0).sum())

print("Number of non-integer values per column (should be 0 after astype(int)):")
display(df_donations[DONATION_AMTS_COLS].map(lambda x: not float(x).is_integer()).sum())

In [ ]:
num_cols_donations = df_donations.select_dtypes(include=['number']).columns

n_cols = 2
n_rows = 4

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, n_rows * 2.5))
axes = axes.flatten()

for i, col in enumerate(num_cols_donations):
    sns.histplot(df_donations[col], discrete=True, ax=axes[i])
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Histograms', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# check if LAST_GIFT_AMT is ever greater than LIFETIME_MAX_GIFT_AMT
last_gift_greater_than_max = df_donations[df_donations['LAST_GIFT_AMT'] > df_donations['LIFETIME_MAX_GIFT_AMT']]

print(f"Number of times LAST_GIFT_AMT is greater than LIFETIME_MAX_GIFT_AMT: {len(last_gift_greater_than_max)}")
if not last_gift_greater_than_max.empty:
    display(last_gift_greater_than_max[['LAST_GIFT_AMT', 'LIFETIME_MAX_GIFT_AMT']].head())

In [ ]:
# for cases where LAST_GIFT_AMT is greater than LIFETIME_MAX_GIFT_AMT, set LIFETIME_MAX_GIFT_AMT to LAST_GIFT_AMT
df_donations.loc[df_donations['LAST_GIFT_AMT'] > df_donations['LIFETIME_MAX_GIFT_AMT'], 'LIFETIME_MAX_GIFT_AMT'] = df_donations['LAST_GIFT_AMT']

# check if there are any instances remaining
last_gift_greater_than_max_after_correction = df_donations[df_donations['LAST_GIFT_AMT'] > df_donations['LIFETIME_MAX_GIFT_AMT']]
print(f"Number of times LAST_GIFT_AMT is greater than LIFETIME_MAX_GIFT_AMT after correction: {len(last_gift_greater_than_max_after_correction)}")

In [ ]:
# check if LAST_GIFT_AMT is ever smaller than LIFETIME_MIN_GIFT_AMT
last_gift_smaller_than_min = df_donations[df_donations['LAST_GIFT_AMT'] < df_donations['LIFETIME_MIN_GIFT_AMT']]

print(f"Number of times LAST_GIFT_AMT is smaller than LIFETIME_MIN_GIFT_AMT: {len(last_gift_smaller_than_min)}")
if not last_gift_smaller_than_min.empty:
    display(last_gift_smaller_than_min[['LAST_GIFT_AMT', 'LIFETIME_MIN_GIFT_AMT']].head())


In [ ]:
# for cases where LAST_GIFT_AMT is smaller than LIFETIME_MIN_GIFT_AMT, set LIFETIME_MIN_GIFT_AMT to LAST_GIFT_AMT
df_donations.loc[df_donations['LAST_GIFT_AMT'] < df_donations['LIFETIME_MIN_GIFT_AMT'], 'LIFETIME_MIN_GIFT_AMT'] = df_donations['LAST_GIFT_AMT']

# check if there are any instances remaining
last_gift_smaller_than_min_after_correction = df_donations[df_donations['LAST_GIFT_AMT'] < df_donations['LIFETIME_MIN_GIFT_AMT']]
print(f"Number of times LAST_GIFT_AMT is smaller than LIFETIME_MIN_GIFT_AMT after correction: {len(last_gift_smaller_than_min_after_correction)}")

In [ ]:
#Cross Validation with df_campaings dataset
#Check cases where lifetime_gift_count == 0 when lifetime_gift_amt > 0
#Assign 1 to lifetime_gift_amount. Since there is a gift amount, then the
#individual donated at least once

temp_df_joined = df_campaigns.join(df_donations)

temp_df_joined.loc[(temp_df_joined['LIFETIME_GIFT_COUNT'] == 0) & (temp_df_joined['LIFETIME_GIFT_AMOUNT'] > 0), "LIFETIME_GIFT_COUNT"] = 1

#Inconsistencies after fix
nr_inconsistencies = temp_df_joined.loc[(temp_df_joined['LIFETIME_GIFT_COUNT'] == 0) & (temp_df_joined['LIFETIME_GIFT_AMOUNT'] > 0)].size
print(f"Number of inconsistencies after correction: {nr_inconsistencies}")


#Join fixed values to df_campaigns dataframe
df_campaigns['LIFETIME_GIFT_COUNT'].update(temp_df_joined['LIFETIME_GIFT_COUNT'])

In [ ]:
temp_df_joined

### 3.2.2. Power Transform (Yeo-Johnson)

Several features are strongly right-skewed.**Yeo-Johnson** power transform is applied, which handles zero and negative values and brings distributions closer to normality — a prerequisite for distance-based clustering algorithms like K-Means.

Applied after transformation, and critically, separately per group. The justification: features across the two groups have completely different units and ranges (home values in hundreds of thousands vs. percentages between 0–100). Without scaling, K-Means would be dominated by whichever feature has the largest absolute range, regardless of its actual informational importance. Min-Max brings everything to [0, 1] so every feature competes on equal footing.

#### Neighbourhood -level Indicators

In [ ]:
# Identify features with |skewness| > 0.75 as candidates for transformation
skewness = df_clean.skew()
skewed_cols = skewness[skewness.abs() > 0.75].index.tolist()
print('Features with |skew| > 0.75 (candidates for power transform):')
print(skewness[skewed_cols].round(3))

In [ ]:
# Apply Yeo-Johnson power transform to skewed features
pt = PowerTransformer(method='yeo-johnson')
df_transformed = df_clean.copy()
df_transformed[skewed_cols] = pt.fit_transform(df_clean[skewed_cols])

print('Skewness after Yeo-Johnson transform:')
print(df_transformed[skewed_cols].skew().round(3))

In [ ]:
# Visual comparison: before vs after for the most skewed features
to_check = [c for c in skewed_cols if c in ALL_COLS]

fig, axes = plt.subplots(2, len(to_check), figsize=(5 * len(to_check), 8))
fig.suptitle('Distribution Before vs After Yeo-Johnson Transform', fontsize=13, fontweight='bold')

for i, col in enumerate(to_check):
    sns.histplot(df_clean[col], kde=True, ax=axes[0, i], color='steelblue')
    axes[0, i].set_title(f'{col}\n(before)', fontsize=9)

    sns.histplot(df_transformed[col], kde=True, ax=axes[1, i], color='seagreen')
    axes[1, i].set_title(f'{col}\n(after)', fontsize=9)

plt.tight_layout()
plt.show()

Applied only to features with |skew| > 0.75 — this threshold-based targeting avoids transforming features that are already reasonably symmetric. Yeo-Johnson specifically is chosen over Box-Cox because it handles zero and negative values (Box-Cox requires strictly positive inputs), which matters here given the data quality issues. The goal is to bring distributions closer to normality, which is a prerequisite for K-Means: Euclidean distance behaves best when feature distributions are roughly symmetric.

<a name='scale'></a>
### 3.2.3. Scaling — Min-Max Normalisation
**Min-Max scaling** is applied to bring all features onto a [0, 1] range, ensuring that features with different units (e.g., home values in $100s vs. percentages) contribute equally to distance calculations. Without scaling, K-Means would be dominated by whichever feature has the largest absolute range, regardless of its actual informational importance.

#### Neighbourhood -level Indicators

In [ ]:
# Separate into the two groups before scaling — each group will have its own scaled version
scaler = MinMaxScaler()

# Group A — Financial
df_financial = pd.DataFrame(
    scaler.fit_transform(df_transformed[FINANCIAL_COLS]),
    columns=FINANCIAL_COLS,
    index=df_transformed.index
)

# Group B — Military
df_military = pd.DataFrame(
    scaler.fit_transform(df_transformed[MILITARY_COLS]),
    columns=MILITARY_COLS,
    index=df_transformed.index
)

print('Group A (Financial) — scaled:')
print(df_financial.describe().round(3))
print()
print('Group B (Military) — scaled:')
print(df_military.describe().round(3))

### 3.2.4. Calculated Variables

#### Campaign & Donation Behaviour

**New Variable 1: Donor Status**

In [ ]:
#Confirm the inconsistency of RECENCY_STATUS_96NK where, for example, Active donors (A) should have given the last gift less than 12 months ago (only)
pd.crosstab(
    df_campaigns['RECENCY_STATUS_96NK'],
    df_campaigns['MONTHS_SINCE_LAST_GIFT']
)

In [ ]:
#Create new variable called DONOR_STATUS to replace RECENCY_STATUS_96NK

conditions = [
    # F: First time
    (df_campaigns['MONTHS_SINCE_FIRST_GIFT'] <= 6) &
    (df_campaigns['LIFETIME_GIFT_COUNT'] == 1),

    # A: Active
    (df_campaigns['MONTHS_SINCE_FIRST_GIFT'] > 12) &
    (df_campaigns['MONTHS_SINCE_LAST_GIFT'] <= 12),

    # E: Inactive
    (df_campaigns['MONTHS_SINCE_LAST_GIFT'] >= 25),

    # L: Lapsing
    (df_campaigns['MONTHS_SINCE_LAST_GIFT'] > 12) &
    (df_campaigns['MONTHS_SINCE_LAST_GIFT'] <= 24),

    # N: New
    (df_campaigns['MONTHS_SINCE_FIRST_GIFT'] <= 12)
]

choices = ['F', 'A', 'E', 'L', 'N']

df_campaigns['DONOR_STATUS'] = np.select(conditions, choices, default='OTHER')

In [ ]:
pd.crosstab(
    df_campaigns['DONOR_STATUS'],
    df_campaigns['MONTHS_SINCE_LAST_GIFT']
)

In [ ]:
pd.crosstab(
    df_campaigns['DONOR_STATUS'],
    df_campaigns['RECENCY_STATUS_96NK']
)


In [ ]:
#Add to Campaigns dataset
df['DONOR_STATUS'] = df_campaigns['DONOR_STATUS']

**New Variable 2: Lifetime Gift per Promo**

In [ ]:
df_campaigns['LIFETIME_GIFT_PER_PROM'] = np.where(
    df_campaigns['LIFETIME_PROM'] > 0,
    df_campaigns['LIFETIME_GIFT_COUNT'] / df_campaigns['LIFETIME_PROM'],
    np.nan   # or 0 depending on your use case
)

In [ ]:
#Add to Campaigns dataset
df['LIFETIME_GIFT_PER_PROM'] = df_campaigns['LIFETIME_GIFT_PER_PROM']

In [ ]:
#Check skewness
df_campaigns['LIFETIME_GIFT_PER_PROM'].skew()

In [ ]:
#Fix for high skewness (>1)
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')

df_campaigns['LIFETIME_GIFT_PER_PROM_PT'] = pt.fit_transform(df_campaigns[['LIFETIME_GIFT_PER_PROM']])

In [ ]:
#Verify results
print("Before:", df_campaigns['LIFETIME_GIFT_PER_PROM'].skew())
print("After:", df_campaigns['LIFETIME_GIFT_PER_PROM_PT'].skew())

**New Variable 3: Months Since First Gift Bin**

In [ ]:
#Create new variable based on bins
bins = [0, 12, 24, 60, 120, df_campaigns['MONTHS_SINCE_FIRST_GIFT'].max()]
labels = ['0-12', '12-24', '24-60', '60-120', '120+']

df_campaigns['MONTHS_SINCE_FIRST_GIFT_BIN'] = pd.cut(df_campaigns['MONTHS_SINCE_FIRST_GIFT'], bins=bins, labels=labels, include_lowest=True)

In [ ]:
#Check distribution
df_campaigns['MONTHS_SINCE_FIRST_GIFT_BIN'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(6, 3))
sns.countplot(x=df_campaigns['MONTHS_SINCE_FIRST_GIFT_BIN'])
plt.title('Distribution of MONTHS_SINCE_FIRST_GIFT (Binned)')
plt.xticks(rotation=45)
plt.show()

#### Donation Amounts

In [ ]:
# create AVG_DONATION based on variable from df_campaigns dataset. This will
#tell us the avg donation that each donor makes

# temp_df_joined = df_campaigns.join(df_donations)
temp_df_joined['LIFETIME_GIFT_COUNT'] = temp_df_joined['LIFETIME_GIFT_COUNT'].astype(float)

# AVG_DONATION = LIFETIME_GIFT_AMOUNT/LIFETIME_GIFT_COUNT. fillna() is used for
#cases where output is nan (Example: dividing 0 by 0)
temp_df_joined['AVG_DONATION'] = temp_df_joined['LIFETIME_GIFT_AMOUNT'].div(temp_df_joined['LIFETIME_GIFT_COUNT']).fillna(0)
temp_df_joined['AVG_DONATION'] = temp_df_joined['AVG_DONATION'].round(decimals=1)

temp_df_joined.info()
display(temp_df_joined.head())

In [ ]:
#Join calculated variable back to df_donations dataframe
df_donations = df_donations.join(temp_df_joined['AVG_DONATION'])

df_donations.info()
df_donations.head()

<a name='reduction'></a>
## 3.3. Data Reduction

### SocioDemographics

WEALTH RATING
Due to the high NULL percentage (46%) and data imputation methods not being reliable, we will drop this column and not use it for clustering. We will instead use Income Group

In [ ]:
#Drop Wealth Rating column
df_socioDemograph = df_socioDemograph.drop(columns=["WEALTH_RATING"])

### Neighbourhood-level Indicators

<a name='corr'></a>
#### Multicollinearity — Correlation Check

In [ ]:
# Check correlations after cleaning on Group A
corr_financial = df_financial.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(corr_financial, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Group A — Post-Processing\nCorrelation Matrix', fontweight='bold')

corr_military = df_military.corr()
sns.heatmap(corr_military, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1], square=True, linewidths=0.5)
axes[1].set_title('Group B — Post-Processing\nCorrelation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

**Observation (Group A):** `MEDIAN_HOME_VALUE`, `MEDIAN_HOUSEHOLD_INCOME`, and `PER_CAPITA_INCOME` remain highly correlated (r > 0.8). This multicollinearity justifies applying PCA to Group A, compressing these three correlated features while retaining `PCT_OWNER_OCCUPIED` as a distinct signal.

**Observation (Group B):** Correlations are moderate — no pair exceeds 0.7 — so PCA is optional here but may still help reduce noise.

<a name='pca'></a>
#### PCA

PCA is applied to Group A to address the high intercorrelation among the three wealth-related features (r > 0.8), which would otherwise over-represent the wealth dimension in Euclidean distance calculations. PCA consolidates the redundant variance into orthogonal components while preserving ≥85% of the total variance."

In Group B the correlations are moderate — no pair exceeds 0.7 — and the four features each represent a conceptually distinct military subpopulation (active military, veterans, Vietnam vets, WW2 vets). Given the absence of meaningful multicollinearity and the interpretability value of retaining original features, PCA is not applied to Group B. The four scaled features will be passed directly into the clustering stage.

In [ ]:
# --- PCA on Group A (Financial Indicators) ---

pca_financial = PCA()
pca_financial.fit(df_financial)

explained_var_A = pca_financial.explained_variance_ratio_
cumulative_var_A = np.cumsum(explained_var_A)

plt.figure(figsize=(6, 4))
plt.bar(range(1, len(explained_var_A)+1), explained_var_A, alpha=0.6, label='Individual', color='steelblue')
plt.plot(range(1, len(cumulative_var_A)+1), cumulative_var_A, marker='o', color='navy', label='Cumulative')
plt.axhline(0.85, linestyle='--', color='grey', label='85% threshold')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Group A — PCA: Explained Variance', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print('Cumulative explained variance by component:')
for i, v in enumerate(cumulative_var_A):
    print(f'  PC{i+1}: {v:.3f}')

In [ ]:
# Choose number of components that explain >= 85% of variance for Group A
n_components_A = int(np.argmax(cumulative_var_A >= 0.85) + 1)
print(f'Number of components chosen for Group A: {n_components_A} ({cumulative_var_A[n_components_A-1]*100:.1f}% variance explained)')

pca_A = PCA(n_components=n_components_A)
pca_A_result = pca_A.fit_transform(df_financial)

df_financial_pca = pd.DataFrame(
    pca_A_result,
    columns=[f'PC_fin_{i+1}' for i in range(n_components_A)],
    index=df_financial.index
)
print('\nGroup A — PCA result (first rows):')
df_financial_pca.head()

In [ ]:
# PCA component loadings for Group A — what does each PC capture?
loadings_A = pd.DataFrame(
    pca_A.components_,
    columns=FINANCIAL_COLS,
    index=[f'PC_fin_{i+1}' for i in range(n_components_A)]
)
print('Group A — PCA Loadings:')
print(loadings_A.round(3))

### Campaigns & Donation Behaviour

In [ ]:
df_campaigns.describe().T

In [ ]:
#Check skewness (above 1 or below -1 is considered excessive)
df_campaigns.select_dtypes(include=['number']).skew()

In [ ]:
import math

count_vars_cpgs = ['LIFETIME_PROM', 'LIFETIME_CARD_PROM', 'RECENT_RESPONSE_COUNT', 'RECENT_CARD_RESPONSE_COUNT', 'NUMBER_PROM_12', 'CARD_PROM_12', 'MONTHS_SINCE_FIRST_GIFT', 'MONTHS_SINCE_LAST_GIFT', 'MONTHS_SINCE_LAST_PROM_RESP', 'LIFETIME_GIFT_COUNT', 'FREQUENCY_STATUS_97NK']

ratio_vars_cpgs = ['RECENT_RESPONSE_PROP', 'RECENT_CARD_RESPONSE_PROP', 'LIFETIME_GIFT_PER_PROM','LIFETIME_GIFT_PER_PROM_PT']

cat_vars_cpgs = ['PEP_STAR', 'RECENT_STAR_STATUS', 'RECENCY_STATUS_96NK', 'DONOR_STATUS', 'MONTHS_SINCE_FIRST_GIFT_BIN']

### Count variables ###
n = len(count_vars_cpgs)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3 * nrows))
axes = axes.flatten()

for i, col in enumerate(count_vars_cpgs):
    sns.histplot(df_campaigns[col], discrete=True, ax=axes[i])
    axes[i].set_title(f"Count: {col}")

for j in range(i + 1, len(axes)):   # hide unused axes
    axes[j].set_visible(False)

fig.suptitle("Count Variable Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### Ratio variables ###
n = len(ratio_vars_cpgs)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3 * nrows))
axes = axes.flatten()

for i, col in enumerate(ratio_vars_cpgs):
    sns.histplot(df_campaigns[col], bins=20, binrange=(0, 1), ax=axes[i])
    axes[i].set_xlim(0, 1)
    axes[i].set_title(f"Ratio: {col}")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Ratio Variable Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### Categorical variables ###
n = len(cat_vars_cpgs)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3 * nrows))
axes = axes.flatten()

for i, col in enumerate(cat_vars_cpgs):
    sns.countplot(x=df_campaigns[col], ax=axes[i])
    axes[i].set_title(f"Categorical: {col}")
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Categorical Variable Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
#Check correrations
num_df = df_campaigns.select_dtypes(include=['number'])
corr = num_df.corr(method='spearman')

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(14, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Spearman Correlation - Numeric Variables')
plt.show()

Drop Columns with many inconsistencies, correlation or skewness

In [ ]:
print("Initial number of variables:", df_campaigns.shape)
df_campaigns.drop('RECENCY_STATUS_96NK', axis = 1, inplace = True) #too many inconsistencies with remaining variable and replaced with new variable donor_status
#df_campaigns.drop('LIFETIME_GIFT_PER_PROM', axis = 1, inplace = True) #high skewness and replaced with power transform version of it
df_campaigns.drop('LIFETIME_CARD_PROM', axis = 1, inplace = True) #high correlation with lifetime_prom(0.93)
print("\nFinal number of variables:", df_campaigns.shape)

In [ ]:
df_campaigns.info()

## Preprocessing Summary

### SocioDemographics

In [ ]:
###THIS IS AN OPTIONAL STEP###

#Convert string value into numeric values. Numeric values are better handled
#by the model for clustering
num_mappings = {
      'DONOR_GENDER': {'U': 0, 'F': 1, 'M': 2},
      'HOME_OWNER': {'U': 0, 'H': 1},
      'URBANICITY': {'?': 0, 'C': 1, 'T':2, 'R': 3, 'S': 4, 'U': 5}
}


#Optional use later after clustering for clarity when interpreting data
txt_mappings = {
      'DONOR_GENDER': {0: 'Unknown', 1: 'Female', 2: 'Male'},
      'HOME_OWNER': {0: 'Unknown', 'Homeowner': 1},
      'URBANICITY': {0: 'Unknown', 1: 'City', 2: 'Rural', 3: 'Suburban', 4: 'Urban'}
}

df_socioDemograph_mapped = df_socioDemograph.copy()

for col, mapping in num_mappings.items():
  df_socioDemograph_mapped[col] = df_socioDemograph[col].map(mapping)

In [ ]:
#FINAL NON-MAPPED DATAFRAMED
print(df_socioDemograph.dtypes)
df_socioDemograph

In [ ]:
#FINAL MAPPED DATAFRAME
print(df_socioDemograph_mapped.dtypes)
df_socioDemograph_mapped

In [ ]:
# Final check
print('Group A — ready for clustering:')
print(df_financial_pca.shape)
print(df_financial_pca.isnull().sum().sum(), 'missing values')

print('\nGroup B — ready for clustering:')
print(df_military.shape)
print(df_military.isnull().sum().sum(), 'missing values')

In [ ]:
df_financial_pca

In [ ]:
df_military

In [ ]:
# Combine both feature groups into a single DataFrame ready for clustering
df_final_neighbourhood = pd.concat([df_financial_pca, df_military], axis=1)

print('Combined DataFrame shape:', df_final_neighbourhood.shape)
print('\nColumns:', df_final_neighbourhood.columns.tolist())
print('\nMissing values:', df_final_neighbourhood.isnull().sum().sum())
df_final_neighbourhood.head()

### Campaign & Donation Behaviour

In [ ]:
df_campaigns.info()
df_campaigns.head()

### Donations Amount

In [ ]:
df_donations.info()
df_donations.head()

# 4. Clustering

## Datasets

In [ ]:
# Datasets to use for clustering:
### SocioDemographics ###
df_socioDemograph_c = df_socioDemograph.copy()

### Neighbourhood-Level Indicators ###
df_neighborhood_levels_c = df_final_neighbourhood.copy()

### Campaign and Donation Behaviour ###
df_campaigns_c = df_campaigns.copy()

### Donations Amount ###
df_donations_c = df_donations.copy()

### FULLY JOINED DATASET ###
df_full_c = df_socioDemograph.join(df_final_neighbourhood.copy())\
                .join(df_campaigns.copy())\
                .join(df_donations.copy())\
                .join(df_financial.copy())

df_full_c.info()

## Kmeans - Response and Amounts

In [ ]:
df_kmeans_ra = df_full_c.copy()

cols = ['LIFETIME_GIFT_AMOUNT',
        'LIFETIME_MAX_GIFT_AMT',
        'LIFETIME_MIN_GIFT_AMT',
        'RECENT_AVG_CARD_GIFT_AMT',
        'RECENT_AVG_GIFT_AMT',
        'AVG_DONATION',
        'LIFETIME_GIFT_COUNT',
        'RECENT_RESPONSE_PROP',
        'RECENT_CARD_RESPONSE_PROP',
        'LIFETIME_GIFT_PER_PROM',
        'FREQUENCY_STATUS_97NK',
        #'NUMBER_PROM_12',
        #'CARD_PROM_12'
      ]

df_donation_amount = df_kmeans_ra[cols]
df_donation_amount

In [ ]:
#Standard Scale Data
mmc = StandardScaler()

df_scaled = mmc.fit_transform(df_donation_amount)

pd.DataFrame(df_scaled).describe()

In [ ]:
# Elbow method: fit KMeans for k=1-10 and append inertia to find optimal clusters
ks = range(1, 11)
inertias = []

for k in ks:
    model = KMeans(n_clusters=k).fit(df_scaled)
    inertias.append(model.inertia_)

In [ ]:
# Plot ks (x-axis) vs inertias (y-axis) using plt.plot().
plt.plot(ks, inertias)

# define the label for the x axis as 'number of clusters' using matplotlib.pyplot.xlabel
plt.xlabel('number of clusters')
# define the label for the y axis as 'inertia' using matplotlib.pyplot.ylabel
plt.ylabel('inertia')
# define the ticks on the x axis using the values of ks
plt.xticks(ks)

plt.show()

In [ ]:
#Execute KMeans Clustering for scaled dataframe
model_k4 = KMeans(n_clusters=4, random_state = 100).fit(df_scaled)
#model_k5 = KMeans(n_clusters=4, random_state = 100).fit(df_scaled)

In [ ]:
df_scaled = pd.DataFrame(df_scaled, columns=df_donation_amount.columns)
df_scaled['label'] = model_k4.labels_

In [ ]:
sns.heatmap(df_scaled.groupby(['label']).mean().transpose(), annot=True, fmt=".2f", cmap="YlOrRd")

In [ ]:
df_donation_amount['label'] = model_k4.labels_
df_donation_amount.groupby(['label']).size()


In [ ]:
df_donation_amount.groupby(['label']).mean().transpose()

Join with Categorical dimensions

In [ ]:
cat_cols = ['DONOR_AGE', 'INCOME_GROUP', 'RECENT_STAR_STATUS', 'PEP_STAR', 'DONOR_STATUS', 'MONTHS_SINCE_FIRST_GIFT_BIN']

df_clustered_dims = df_donation_amount.join(df_full_c[cat_cols])
df_clustered_dims[['DONOR_AGE', 'INCOME_GROUP', 'RECENT_STAR_STATUS', 'PEP_STAR', 'DONOR_STATUS', 'MONTHS_SINCE_FIRST_GIFT_BIN', 'label']].groupby(['label']).describe().T

In [ ]:
df_clustered_dims['MONTHS_SINCE_FIRST_GIFT_BIN'] = df_clustered_dims['MONTHS_SINCE_FIRST_GIFT_BIN'].astype(str)
df_clustered_dims[['DONOR_STATUS', 'MONTHS_SINCE_FIRST_GIFT_BIN', 'label']].groupby(['label']).describe(include='O').T

### Notes on individual clusters

**Cluster 0 — Responsive Donors but small donation amounts**

**Profile:** These donors have very high engagement with us and donate with moderate-to-high frequency. This can be verified by the Frequency_Status_97NK mean that tells us that they donate 3 times a year on average. Also, the Lifetime_Gift_Count is the second highest of all clusters.

**Approach to be taken:** Keep contacting these clients in a similar manner but give some incentive to increase the donation amount

**Cluster 1 — Non-Frequent and slightly unresponsive but high donation amounts**

**Profile:** High-potential lapsed donors — they have the highest average donation but the lowest number of donations.

**Approach to be taken:** These might be worth investing on a more personalized approach since they have high potential to give a big donation. Also, since this is a smaller cluster, it is easier to plan a more personalized and persuasive approach. On another note, many of these donors donated for the first time around 2 years ago, so they are quite recent and might need a reason in seeing the benefit of becoming long time donators.

**Cluster 2 — Average Donors that are less enagaged**

**Profile:** These are the majority of our donors (~40%). They donate an average amount, have low response to our campaigns and donate very few times. These are potentially one-time donors.

**Approach to be taken:** Since these are probably one-time donors that haven't donated in a long time (This can be verified by the low mean in RECENT_STAR_STATUS, low mean in LIFETIME_GIFT_COUNT and long time since first donation), a reactivation campaign might be the best approach. They should not be cut from campagins since they are the majority of our donors.

**Cluster 3 — Best Overall Donors**

**Profile:** Our best donors — high frequency and amounts in donations and quite active in responses

**Approach to be taken:** These are our longtime donors that do not need much engagement, just a gentle reminder to make their usual donation as always. They should be treated with the utmost respect



### Notes applicable to all clusters

**Donor Age:** Mean of Donor Age is very similar accross all clusters (50-60 years old). So all donors should be contacted in a very respectfull and adult manner, and not at all in a way intended for youthfull audiences

**Income Group**: Mean of Income Group is also very similar accross all groups (Between Income Group 3 and 4). This tells us that the income of our donors doesn't necessarily dictate how much they donate.

### Notes on attempts made for clustering

A first attempt of clustering with k=5 was made but we saw that 2 clusters were very similar, the only difference being that one donated quite more to card solicitations. So we, reduced to k=4 and verified that these 2 similar clusters were merged into our current **cluster 2**.

An attempt was made to include the number of promotions **(NUMBER_PROM_12 & CARD_PROM_12)** sent in the last 12 months, but we saw very little variance between clusters. The conclusion we take from these variables is that the campaigns being performed previously were not personalized and every type of promotion/campaigns/solicitation was sent to everyone.



# Action Plan

Based on the characteristics of clusters we have the following recommendations for future promotions:


1) Highly customized approach for Cluster 3 with personalized emails, exclusive newsletters, occasional calls or hardwritten notes, and recognition in the community.


2) Customized approach for Cluster 1 focused on retaining and building a long-term relationship.


3) Automated approaches for Cluster 0 such as email campaigns and website prompts, while trying to upsell and increase the donation size.


4) Low cost approaches for Cluster 2 with bulk email campaigns focused on maximizing the reach, scaling and reactivation
